Bu notebook’ta:
Kendi Multiclass Logistic Regression’imizi saf Python ile implemente ediyoruz.
Üç dataset için ayrı model eğitiyoruz:
- PlantVillage
- Plant Disease Detection
- PlantDoc Converted
Sonra metrikleri hesaplıyoruz:
Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC
(burada sklearn.metrics sadece değerlendirme için, eğitimde hiç kullanılmıyor).
Modelleri .npz dosyaları olarak kaydediyoruz.


# Multiclass Logistic Regression Training (Plant Datasets)

Bu notebook, sıfırdan implemente edilmiş Multiclass Logistic Regression
modelini kullanarak üç farklı veri seti üzerinde eğitim yapar:

1. PlantVillage (`preprocessed_plantvillage`)
2. Plant Disease Detection (`preprocessed_pdd`)
3. PlantDoc (converted) (`preprocessed_plantdoc`)

Eğitim saf Python ile yapılmaktadır. `sklearn.metrics` sadece sonuçları
değerlendirmek ve doğrulamak için kullanılmaktadır.


# Cell 1- Importlar

In [ ]:
import os
import math
import random
import numpy as np

# Only for evaluation/verification (not for training)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)


# Cell 2 – Yardımcı fonksiyonlar (softmax, batch iteration)

In [ ]:
def softmax(logits):
    """
    Compute softmax probabilities from a list of logits.
    """
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    s = sum(exps)
    if s == 0.0:
        c = len(logits)
        return [1.0 / c for _ in range(c)]
    return [e / s for e in exps]

def cross_entropy_loss(probs, true_class):
    """
    Compute cross-entropy loss for a single sample:
    L = -log( p_true_class )
    """
    p = probs[true_class]
    # numerical stability
    if p < 1e-15:
        p = 1e-15
    return -math.log(p)



# Cell 3 – Multiclass Logistic Regression sınıfı

In [ ]:
class MulticlassLogisticRegression:
    """
    Multiclass Logistic Regression implemented from scratch using
    Python lists and basic math operations (no ML libraries).
    """

    def __init__(self, num_features, num_classes, learning_rate=0.1):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        # Small random initialization for weights and biases
        self.W = [
            [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]
            for _ in range(num_features)
        ]
        self.b = [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]

    def _compute_logits(self, x):
        """
        Compute logits for a single sample x.
        x is a list of length num_features.
        Returns list of length num_classes.
        """
        logits = [0.0 for _ in range(self.num_classes)]
        for k in range(self.num_classes):
            s = 0.0
            for j in range(self.num_features):
                s += x[j] * self.W[j][k]
            s += self.b[k]
            logits[k] = s
        return logits

    def predict_proba_one(self, x):
        """
        Predict class probabilities for a single sample x.
        """
        logits = self._compute_logits(x)
        probs = softmax(logits)
        return probs

    def predict_one(self, x):
        """
        Predict class index for a single sample x.
        """
        probs = self.predict_proba_one(x)
        best_class = 0
        best_prob = probs[0]
        for k in range(1, self.num_classes):
            if probs[k] > best_prob:
                best_prob = probs[k]
                best_class = k
        return best_class

    def predict(self, X):
        """
        Predict class indices for a list of samples X.
        Each element in X is a feature vector (list).
        """
        return [self.predict_one(x) for x in X]

    def predict_proba(self, X):
        """
        Predict probability distributions for a list of samples X.
        Returns a list of lists (N x num_classes).
        """
        return [self.predict_proba_one(x) for x in X]

    def fit(self, X_train, y_train, num_epochs=10, X_val=None, y_val=None, verbose=True):
        """
        Train the model using online (sample-by-sample) gradient descent.

        X_train: list (veya numpy array) of feature vectors
        y_train: list (veya numpy array) of integer labels
        """
        n_samples = len(X_train)

        for epoch in range(num_epochs):
            # Her epoch başında indexleri karıştır (shuffle)
            indices = list(range(n_samples))
            random.shuffle(indices)

            total_loss = 0.0
            correct = 0

            # Yüzde göstermek için 10 parçaya bölelim
            progress_step = max(1, n_samples // 10)

            for step, idx in enumerate(indices):
                x = X_train[idx]
                true_class = y_train[idx]

                # ---- Forward ----
                logits = self._compute_logits(x)
                probs = softmax(logits)
                loss = cross_entropy_loss(probs, true_class)
                total_loss += loss

                # Doğru / yanlış say (train_acc için)
                pred_class = 0
                best_prob = probs[0]
                for k in range(1, self.num_classes):
                    if probs[k] > best_prob:
                        best_prob = probs[k]
                        pred_class = k
                if pred_class == true_class:
                    correct += 1

                # ---- Gradient + Güncelleme ----
                for k in range(self.num_classes):
                    if k == true_class:
                        error_k = probs[k] - 1.0
                    else:
                        error_k = probs[k]

                    for j in range(self.num_features):
                        grad_w_jk = error_k * x[j]
                        self.W[j][k] -= self.learning_rate * grad_w_jk

                    grad_b_k = error_k
                    self.b[k] -= self.learning_rate * grad_b_k

                # ---- İlerleme yüzdesi ----
                if verbose and ((step + 1) % progress_step == 0 or (step + 1) == n_samples):
                    pct = (step + 1) / n_samples * 100.0
                    print(
                        f"\rEpoch {epoch + 1}/{num_epochs} - "
                        f"{pct:5.1f}% completed",
                        end="",
                        flush=True
                    )

            # Epoch bitti, satırı kır
            if verbose:
                print()

            # ---- Epoch sonunda loss + accuracy ----
            avg_loss = total_loss / n_samples
            train_acc = correct / n_samples

            val_acc = None
            if X_val is not None and y_val is not None and len(X_val) > 0:
                correct_val = 0
                for x_val, y_true_val in zip(X_val, y_val):
                    y_pred_val = self.predict_one(x_val)
                    if y_pred_val == y_true_val:
                        correct_val += 1
                val_acc = correct_val / len(X_val)

            if verbose:
                if val_acc is not None:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f} - "
                        f"val_acc: {val_acc:.4f}"
                    )
                else:
                    print(
                        f"Epoch {epoch + 1}/{num_epochs} - "
                        f"loss: {avg_loss:.4f} - "
                        f"train_acc: {train_acc:.4f}"
                    )
        

    def save(self, path):
        """
        Save model parameters (W, b, num_features, num_classes)
        to a .npz file.
        """
        W_array = np.array(self.W, dtype=np.float32)
        b_array = np.array(self.b, dtype=np.float32)
        np.savez(
            path,
            W=W_array,
            b=b_array,
            num_features=self.num_features,
            num_classes=self.num_classes,
        )
        print(f"Model saved to {path}")

    @classmethod
    def load(cls, path):
        """
        Load model parameters from a .npz file and return a new instance.
        """
        data = np.load(path)

        num_features = int(data["num_features"])
        num_classes = int(data["num_classes"])

        model = cls(
            num_features=num_features,
            num_classes=num_classes,
            learning_rate=0.01  # learning_rate not critical for prediction
        )

        W_array = data["W"]
        b_array = data["b"]

        model.W = W_array.tolist()
        model.b = b_array.tolist()

        print(f"Model loaded from {path}")
        print("num_features:", num_features)
        print("num_classes:", num_classes)

        return model



# Cell 4 – Değerlendirme fonksiyonu (metrics)

In [ ]:
def evaluate_model(model, X, y, average="macro", dataset_name=""):
    """
    Evaluate a trained model on given data and print metrics:
    Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC.
    """
    # Convert X to list of lists if it is a numpy array
    if isinstance(X, np.ndarray):
        X_list = X.tolist()
    else:
        X_list = X

    y_true = np.array(y)

    # Predictions
    y_pred = np.array(model.predict(X_list))

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    # Probabilities for ROC-AUC
    y_proba = np.array(model.predict_proba(X_list))
    # Multi-class ROC-AUC (one-vs-rest)
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average=average)
    except ValueError:
        roc_auc = float("nan")

    print("\n" + "=" * 60)
    if dataset_name:
        print(f"Evaluation on {dataset_name}")
    else:
        print("Evaluation")

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc_auc,
        "confusion_matrix": cm,
    }



# Cell 5 – Preprocessed veriyi yükle

In [ ]:
# Paths for preprocessed datasets (relative to this notebook)

PV_DIR = "preprocessed_plantvillage"
PDD_DIR = "preprocessed_pdd"
PD_DIR = "preprocessed_plantdoc"


def load_preprocessed_dataset(base_dir):
    """
    Load train/val/test numpy arrays and class names from a preprocessed directory.
    """
    train_X = np.load(os.path.join(base_dir, "train_X.npy"))
    train_y = np.load(os.path.join(base_dir, "train_y.npy"))
    val_X = np.load(os.path.join(base_dir, "val_X.npy"))
    val_y = np.load(os.path.join(base_dir, "val_y.npy"))
    test_X = np.load(os.path.join(base_dir, "test_X.npy"))
    test_y = np.load(os.path.join(base_dir, "test_y.npy"))

    class_names_path = os.path.join(base_dir, "class_names.txt")
    class_names = []
    with open(class_names_path, "r", encoding="utf-8") as f:
        for line in f:
            class_names.append(line.strip())

    print(f"Loaded dataset from '{base_dir}'")
    print("Train:", train_X.shape, "Val:", val_X.shape, "Test:", test_X.shape)
    print("Number of classes:", len(class_names))
    return train_X, train_y, val_X, val_y, test_X, test_y, class_names



# Cell 6 – PlantVillage için modeli eğit

In [ ]:
# ----- PlantVillage -----
X_train_pv, y_train_pv, X_val_pv, y_val_pv, X_test_pv, y_test_pv, class_names_pv = load_preprocessed_dataset(PV_DIR)

num_features_pv = X_train_pv.shape[1]
num_classes_pv = len(class_names_pv)

# Convert to Python lists for our pure Python implementation
X_train_pv_list = X_train_pv.tolist()
X_val_pv_list = X_val_pv.tolist()
X_test_pv_list = X_test_pv.tolist()
y_train_pv_list = y_train_pv.tolist()
y_val_pv_list = y_val_pv.tolist()
y_test_pv_list = y_test_pv.tolist()

learning_rate_pv = 0.1
num_epochs_pv = 20  # you can tune this

model_pv = MulticlassLogisticRegression(
    num_features=num_features_pv,
    num_classes=num_classes_pv,
    learning_rate=learning_rate_pv,
)

model_pv.fit(
    X_train_pv_list,
    y_train_pv_list,
    num_epochs=num_epochs_pv,
    X_val=X_val_pv_list,
    y_val=y_val_pv_list,
    verbose=True,
)

# Evaluate on train/val/test
metrics_pv_train = evaluate_model(model_pv, X_train_pv, y_train_pv, dataset_name="PlantVillage - Train")
metrics_pv_val = evaluate_model(model_pv, X_val_pv, y_val_pv, dataset_name="PlantVillage - Val")
metrics_pv_test = evaluate_model(model_pv, X_test_pv, y_test_pv, dataset_name="PlantVillage - Test")

# Save model
os.makedirs("models", exist_ok=True)
model_pv.save("models/logreg_plantvillage.npz")



Not: num_epochs_pv ve learning_rate_pv’yi eğitimin hızına/performansına göre değiştirebilirsin.

# Cell 7 – Plant Disease Detection için model

In [ ]:
# ----- Plant Disease Detection -----
X_train_pdd, y_train_pdd, X_val_pdd, y_val_pdd, X_test_pdd, y_test_pdd, class_names_pdd = load_preprocessed_dataset(PDD_DIR)

num_features_pdd = X_train_pdd.shape[1]
num_classes_pdd = len(class_names_pdd)

X_train_pdd_list = X_train_pdd.tolist()
X_val_pdd_list = X_val_pdd.tolist()
X_test_pdd_list = X_test_pdd.tolist()
y_train_pdd_list = y_train_pdd.tolist()
y_val_pdd_list = y_val_pdd.tolist()
y_test_pdd_list = y_test_pdd.tolist()

learning_rate_pdd = 0.1
num_epochs_pdd = 20  # tune if needed

model_pdd = MulticlassLogisticRegression(
    num_features=num_features_pdd,
    num_classes=num_classes_pdd,
    learning_rate=learning_rate_pdd,
)

model_pdd.fit(
    X_train_pdd_list,
    y_train_pdd_list,
    num_epochs=num_epochs_pdd,
    X_val=X_val_pdd_list,
    y_val=y_val_pdd_list,
    verbose=True,
)

metrics_pdd_train = evaluate_model(model_pdd, X_train_pdd, y_train_pdd, dataset_name="PDD - Train")
metrics_pdd_val = evaluate_model(model_pdd, X_val_pdd, y_val_pdd, dataset_name="PDD - Val")
metrics_pdd_test = evaluate_model(model_pdd, X_test_pdd, y_test_pdd, dataset_name="PDD - Test")

model_pdd.save("models/logreg_plantdiseasedetection.npz")



# Cell 8 – PlantDoc Converted için model

In [ ]:
# ----- PlantDoc (Converted) -----
X_train_pd, y_train_pd, X_val_pd, y_val_pd, X_test_pd, y_test_pd, class_names_pd = load_preprocessed_dataset(PD_DIR)

num_features_pd = X_train_pd.shape[1]
num_classes_pd = len(class_names_pd)

X_train_pd_list = X_train_pd.tolist()
X_val_pd_list = X_val_pd.tolist()
X_test_pd_list = X_test_pd.tolist()
y_train_pd_list = y_train_pd.tolist()
y_val_pd_list = y_val_pd.tolist()
y_test_pd_list = y_test_pd.tolist()

learning_rate_pd = 0.1
num_epochs_pd = 15  # tune if needed

model_pd = MulticlassLogisticRegression(
    num_features=num_features_pd,
    num_classes=num_classes_pd,
    learning_rate=learning_rate_pd,
)

model_pd.fit(
    X_train_pd_list,
    y_train_pd_list,
    num_epochs=num_epochs_pd,
    X_val=X_val_pd_list,
    y_val=y_val_pd_list,
    verbose=True,
)

metrics_pd_train = evaluate_model(model_pd, X_train_pd, y_train_pd, dataset_name="PlantDoc Conv - Train")
metrics_pd_val = evaluate_model(model_pd, X_val_pd, y_val_pd, dataset_name="PlantDoc Conv - Val")
metrics_pd_test = evaluate_model(model_pd, X_test_pd, y_test_pd, dataset_name="PlantDoc Conv - Test")

model_pd.save("models/logreg_plantdoc_converted.npz")

